In [ ]:
# %% [markdown]
# # 1. EDA - Exploratory Data Analysis
# ## Dataset: Online Shoppers Purchasing Intention

# %% [code]
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import DataLoader
from src.visualization import DataVisualizer

# Cấu hình hiển thị
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# %% [code]
# Load dữ liệu
loader = DataLoader('../data/raw/online_shoppers_intention.csv')
df = loader.load_data()

# %% [markdown]
# ### 1.1. Tổng quan dữ liệu

# %% [code]
print("=== THÔNG TIN DỮ LIỆU ===")
print(f"Shape: {df.shape}")
print("\n=== 5 DÒNG ĐẦU TIÊN ===")
df.head()

# %% [code]
print("\n=== KIỂU DỮ LIỆU ===")
df.info()

# %% [code]
print("\n=== THỐNG KÊ MÔ TẢ (NUMERIC) ===")
df.describe()

# %% [code]
print("\n=== THỐNG KÊ MÔ TẢ (CATEGORICAL) ===")
categorical_cols = ['Month', 'VisitorType', 'Weekend', 'Revenue']
for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].value_counts())
    print("-" * 30)

# %% [markdown]
# ### 1.2. Phân tích biến mục tiêu (Revenue)

# %% [code]
# Phân phối của biến mục tiêu
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
revenue_counts = df['Revenue'].value_counts()
colors = ['#ff6b6b', '#4ecdc4']
axes[0].pie(revenue_counts.values, labels=['Không mua (False)', 'Mua (True)'], 
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[0].set_title('Tỷ lệ khách hàng mua hàng', fontsize=14, fontweight='bold')

# Bar chart
sns.barplot(x=revenue_counts.index, y=revenue_counts.values, ax=axes[1], palette=colors)
axes[1].set_xlabel('Revenue')
axes[1].set_ylabel('Số lượng')
axes[1].set_title('Số lượng khách hàng theo nhóm', fontsize=14, fontweight='bold')
for i, v in enumerate(revenue_counts.values):
    axes[1].text(i, v + 50, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../results/figures/revenue_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# %% [markdown]
# **Nhận xét:** Dữ liệu mất cân bằng nghiêm trọng - chỉ 15.5% khách hàng mua hàng. Cần xử lý imbalance.

# %% [markdown]
# ### 1.3. Phân tích các thuộc tính số

# %% [code]
numeric_cols = ['Administrative', 'Administrative_Duration', 'Informational', 
                'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
                'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay']

# Boxplot so sánh giữa 2 nhóm
fig, axes = plt.subplots(2, 5, figsize=(20, 10))
axes = axes.ravel()

for i, col in enumerate(numeric_cols):
    data = [df[df['Revenue']==False][col].dropna(), 
            df[df['Revenue']==True][col].dropna()]
    
    bp = axes[i].boxplot(data, labels=['Không mua', 'Mua'], patch_artist=True)
    bp['boxes'][0].set_facecolor('#ff6b6b')
    bp['boxes'][1].set_facecolor('#4ecdc4')
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_ylabel('Giá trị')
    axes[i].grid(True, alpha=0.3)

plt.suptitle('So sánh phân phối các thuộc tính số giữa 2 nhóm', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/numeric_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# %% [code]
# Correlation matrix
numeric_df = df[numeric_cols + ['Revenue']]
# Chuyển Revenue thành số
numeric_df['Revenue_num'] = numeric_df['Revenue'].map({False: 0, True: 1})

corr_matrix = numeric_df.corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Ma trận tương quan giữa các thuộc tính số', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# %% [markdown]
# ### 1.4. Phân tích thuộc tính phân loại

# %% [code]
categorical_cols = ['Month', 'VisitorType', 'Weekend']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(categorical_cols):
    # Tạo crosstab
    ct = pd.crosstab(df[col], df['Revenue'], normalize='index') * 100
    
    ct.plot(kind='bar', ax=axes[i], color=['#ff6b6b', '#4ecdc4'])
    axes[i].set_title(f'Tỷ lệ mua hàng theo {col}', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Tỷ lệ (%)')
    axes[i].legend(['Không mua', 'Mua'])
    axes[i].grid(True, alpha=0.3, axis='y')
    
    # Thêm giá trị lên bars
    for container in axes[i].containers:
        axes[i].bar_label(container, fmt='%.1f%%')

plt.tight_layout()
plt.savefig('../results/figures/categorical_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# %% [markdown]
# ### 1.5. Phát hiện ngoại lệ

# %% [code]
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return len(outliers), lower_bound, upper_bound

outlier_info = {}
for col in numeric_cols:
    count, lb, ub = detect_outliers_iqr(df, col)
    outlier_info[col] = {
        'outliers_count': count,
        'outliers_percent': (count/len(df))*100,
        'lower_bound': lb,
        'upper_bound': ub
    }

outlier_df = pd.DataFrame(outlier_info).T
print("=== PHÁT HIỆN NGOẠI LỆ (IQR METHOD) ===")
print(outlier_df.sort_values('outliers_percent', ascending=False))

# %% [code]
# Visualization outliers
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.ravel()

for i, col in enumerate(numeric_cols):
    axes[i].boxplot([df[df['Revenue']==False][col], df[df['Revenue']==True][col]], 
                    labels=['Không mua', 'Mua'], patch_artist=True)
    axes[i].set_title(col, fontweight='bold')
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Phát hiện ngoại lệ trong dữ liệu', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/outliers_detection.png', dpi=300, bbox_inches='tight')
plt.show()